# MobileNetV4 transfer-learning baseline (`mnv4-001`)

PyTorch+timm, ImageNet pretrained MobileNetV4 Conv-S, frozen backbone. Data protocol được lấy từ artifact `cnn-001`: cùng archive processed, canonical deduplication và participant split.


In [ ]:
%pip -q install 'timm==1.0.27' 'huggingface-hub>=0.25' safetensors pandas scikit-learn matplotlib seaborn


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, os, random, zipfile
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt, seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
from timm.data import resolve_model_data_config
from huggingface_hub import hf_hub_download, snapshot_download
from safetensors.torch import load_file, save_file
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

EXPERIMENT_ID = 'mnv4-001-participant-disjoint-frozen'
DATASET_REPO, DATASET_REVISION = 'hnam25/asl-hand-gesture-images', '8f36ac00ece6dfce94410a980a839d93a912d366'
PROCESSED_ARCHIVE = 'ASL_HG_36000/ASL_Processed_Images.zip'
CNN_ARTIFACT_REPO, CNN_ARTIFACT_REVISION = 'hnam25/asl-hg-cnn-baseline', '064551d5634ef34d3e6a9132ab3d4a374b2f5633'
TIMM_REPO, TIMM_REVISION, MODEL_NAME = 'timm/mobilenetv4_conv_small.e2400_r224_in1k', '331fb803779522b685cf942e15f914fb6741c1eb', 'mobilenetv4_conv_small.e2400_r224_in1k'
SEED, BATCH_SIZE, EPOCHS, LEARNING_RATE, WEIGHT_DECAY = 42, 64, 20, 1e-3, 1e-4
CLASSES = [str(i) for i in range(10)] + [chr(i) for i in range(ord('A'), ord('Z') + 1)]
ROOT = Path('/content/asl-mnv4-baseline'); HF_ROOT, CNN_ROOT, PROCESSED, OUTPUTS = ROOT/'hf', ROOT/'cnn-artifact', ROOT/'processed', ROOT/'outputs'
for directory in (HF_ROOT, CNN_ROOT, PROCESSED, OUTPUTS/'models', OUTPUTS/'metrics', OUTPUTS/'figures', OUTPUTS/'logs', OUTPUTS/'metadata'): directory.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'experiment_id': EXPERIMENT_ID, 'torch': torch.__version__, 'timm': timm.__version__, 'device': str(DEVICE)})


In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()

snapshot_download(repo_id=DATASET_REPO, repo_type='dataset', revision=DATASET_REVISION, local_dir=HF_ROOT, allow_patterns=[PROCESSED_ARCHIVE])
snapshot_download(repo_id=CNN_ARTIFACT_REPO, revision=CNN_ARTIFACT_REVISION, local_dir=CNN_ROOT, allow_patterns=['metadata/deduplication_manifest.csv', 'metadata/split_manifest.json', 'metadata/experiment_config.json'])
archive = HF_ROOT/PROCESSED_ARCHIVE
cnn_config = json.loads((CNN_ROOT/'metadata'/'experiment_config.json').read_text())
if sha256_file(archive) != cnn_config['processed_archive_sha256']: raise RuntimeError('Processed archive SHA does not match cnn-001 provenance.')
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (PROCESSED/member.filename).resolve()
        if PROCESSED.resolve() not in target.parents and target != PROCESSED.resolve(): raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
    z.extractall(PROCESSED)
print('Processed archive verified:', cnn_config['processed_archive_sha256'])


In [ ]:
# Reconstruct exactly the canonical records and participant partition used by cnn-001.
dedup = pd.read_csv(CNN_ROOT/'metadata'/'deduplication_manifest.csv')
split_manifest = json.loads((CNN_ROOT/'metadata'/'split_manifest.json').read_text())
dedup['is_canonical'] = dedup.is_canonical.astype(bool)
canonical = dedup[dedup.is_canonical].copy()
path_by_relative = {}
for path in PROCESSED.rglob('*'):
    if path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}: continue
    label = next((part for part in reversed(path.parts[:-1]) if part in CLASSES), None)
    if label: path_by_relative[f'{label}/{path.name}'] = str(path)
canonical['image_path'] = canonical.relative_path.map(path_by_relative)
if canonical.image_path.isna().any(): raise RuntimeError('Processed archive does not contain every cnn-001 canonical image.')
parts = split_manifest['participants']
train = canonical[canonical.participant.isin(parts['train'])].copy(); validation = canonical[canonical.participant.eq(parts['validation'])].copy(); test = canonical[canonical.participant.eq(parts['test'])].copy()
if set(train.sha256) & set(validation.sha256) or set(train.sha256) & set(test.sha256) or set(validation.sha256) & set(test.sha256): raise RuntimeError('Hash leakage.')
for name, frame in {'train': train, 'validation': validation, 'test': test}.items(): frame[['relative_path', 'label', 'participant', 'sha256']].to_csv(OUTPUTS/'metadata'/f'{name}.csv', index=False)
(OUTPUTS/'metadata'/'split_manifest.json').write_text(json.dumps(split_manifest, indent=2), encoding='utf-8')
(OUTPUTS/'metadata'/'deduplication_manifest.csv').write_text((CNN_ROOT/'metadata'/'deduplication_manifest.csv').read_text(), encoding='utf-8')
print({'train': len(train), 'validation': len(validation), 'test': len(test), 'participants': parts})


In [ ]:
# Load the pinned ImageNet-1k safetensors checkpoint, reset the classifier, and freeze the pretrained backbone.
weight_path = hf_hub_download(repo_id=TIMM_REPO, filename='model.safetensors', revision=TIMM_REVISION)
weight_sha256 = sha256_file(weight_path)
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=1000)
state_dict = load_file(weight_path); incompatible = model.load_state_dict(state_dict, strict=True)
assert not incompatible.missing_keys and not incompatible.unexpected_keys
for parameter in model.parameters(): parameter.requires_grad = False
model.reset_classifier(num_classes=len(CLASSES))
for parameter in model.get_classifier().parameters(): parameter.requires_grad = True
model = model.to(DEVICE)
data_config = resolve_model_data_config(model)
transform = transforms.Compose([transforms.Resize(tuple(data_config['input_size'][1:])), transforms.ToTensor(), transforms.Normalize(mean=data_config['mean'], std=data_config['std'])])
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
print({'weight_sha256': weight_sha256, 'input_size': data_config['input_size'], 'trainable_parameters': trainable_parameters, 'total_parameters': total_parameters})


In [ ]:
class ASLDataset(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True); self.label_index = {label: index for index, label in enumerate(CLASSES)}
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image: tensor = transform(image.convert('RGB'))
        return tensor, self.label_index[row.label]

generator = torch.Generator().manual_seed(SEED)
loaders = {name: DataLoader(ASLDataset(frame), batch_size=BATCH_SIZE, shuffle=(name=='train'), num_workers=2, pin_memory=True, generator=generator) for name, frame in {'train': train, 'validation': validation, 'test': test}.items()}
criterion = nn.CrossEntropyLoss(); optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY); scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=.2)
checkpoint = OUTPUTS/'models'/'mnv4_001_participant_disjoint_frozen.safetensors'
history, best_val_loss, stale_epochs = [], float('inf'), 0
def set_frozen_batchnorm_eval(network):
    for layer in network.modules():
        if isinstance(layer, nn.modules.batchnorm._BatchNorm): layer.eval()

for epoch in range(1, EPOCHS + 1):
    model.train(); set_frozen_batchnorm_eval(model); train_loss = train_correct = train_total = 0
    for images, labels in loaders['train']:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(); logits = model(images); loss = criterion(logits, labels); loss.backward(); optimizer.step()
        train_loss += loss.item() * len(labels); train_correct += (logits.argmax(1) == labels).sum().item(); train_total += len(labels)
    model.eval(); val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for images, labels in loaders['validation']:
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True); logits = model(images); loss = criterion(logits, labels)
            val_loss += loss.item() * len(labels); val_correct += (logits.argmax(1) == labels).sum().item(); val_total += len(labels)
    row = {'epoch': epoch, 'loss': train_loss/train_total, 'accuracy': train_correct/train_total, 'val_loss': val_loss/val_total, 'val_accuracy': val_correct/val_total, 'learning_rate': optimizer.param_groups[0]['lr']}; history.append(row); scheduler.step(row['val_loss']); print(row, flush=True)
    if row['val_loss'] < best_val_loss:
        best_val_loss, stale_epochs = row['val_loss'], 0; save_file({key: value.detach().cpu().contiguous() for key, value in model.state_dict().items()}, str(checkpoint))
    else:
        stale_epochs += 1
        if stale_epochs >= 5: print('Early stopping'); break
pd.DataFrame(history).to_csv(OUTPUTS/'logs'/'training_history.csv', index=False)


In [ ]:
model.load_state_dict(load_file(checkpoint)); model.eval(); truth, predicted = [], []
with torch.no_grad():
    for images, labels in loaders['test']:
        logits = model(images.to(DEVICE, non_blocking=True)); predicted.extend(logits.argmax(1).cpu().tolist()); truth.extend(labels.tolist())
report = classification_report(truth, predicted, labels=range(36), target_names=CLASSES, output_dict=True, zero_division=0); matrix = confusion_matrix(truth, predicted, labels=range(36))
pd.DataFrame(report).T.to_csv(OUTPUTS/'metrics'/'classification_report.csv'); pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(OUTPUTS/'metrics'/'confusion_matrix.csv')
mapping = {label: index for index, label in enumerate(CLASSES)}; o, zero = mapping['O'], mapping['0']
summary = {'experiment_id': EXPERIMENT_ID, 'test_accuracy': float(accuracy_score(truth, predicted)), 'macro_precision': report['macro avg']['precision'], 'macro_recall': report['macro avg']['recall'], 'macro_f1': report['macro avg']['f1-score'], 'O_recall': report['O']['recall'], '0_recall': report['0']['recall'], 'O_to_0': int(matrix[o, zero]), '0_to_O': int(matrix[zero, o]), 'best_validation_accuracy': max(row['val_accuracy'] for row in history), 'epochs_ran': len(history)}
(OUTPUTS/'metrics'/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
plt.figure(figsize=(16,13)); sns.heatmap(matrix, cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.savefig(OUTPUTS/'figures'/'confusion_matrix.png', dpi=180); plt.close()
config = {'experiment_id': EXPERIMENT_ID, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'dataset_repo': DATASET_REPO, 'dataset_revision': DATASET_REVISION, 'processed_archive': PROCESSED_ARCHIVE, 'processed_archive_sha256': cnn_config['processed_archive_sha256'], 'cnn_artifact_repo': CNN_ARTIFACT_REPO, 'cnn_artifact_revision': CNN_ARTIFACT_REVISION, 'timm_repo': TIMM_REPO, 'timm_revision': TIMM_REVISION, 'timm_model_name': MODEL_NAME, 'timm_weight_file': 'model.safetensors', 'timm_weight_sha256': weight_sha256, 'classes': CLASSES, 'split_manifest': split_manifest, 'training': {'frozen_backbone': True, 'batch_size': BATCH_SIZE, 'epochs_max': EPOCHS, 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY, 'total_parameters': total_parameters, 'trainable_parameters': trainable_parameters}, 'torch': torch.__version__, 'timm': timm.__version__}
(OUTPUTS/'metadata'/'experiment_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
os.system(f"pip freeze > {OUTPUTS/'metadata'/'environment.txt'}")
print(json.dumps(summary, indent=2))


## Publish

Download all `outputs/`, create `hnam25/asl-hg-mobilenetv4-baseline`, upload notebook + artifacts, then download checkpoint again and compare SHA-256.
